# NeerNiti — Module 3 Verification & Monitoring Training Pipeline
### Time-Series Forecasting for Groundwater Levels (Bi-LSTM-Attn)

This notebook trains the **NeerNiti Module 3 Groundwater Verification Engine**.

---

### Key Technical Innovations:
1. **Bi-LSTM with Attention** — Learns long-term seasonal trends and short-term volatility in groundwater levels.
2. **Exogenous Regressors** — Integrates monsoon rainfall data to inform recharge peaks.
3. **Walkforward Cross-Validation** — Rolling-origin temporal CV to strictly prevent data leakage.
4. **Optuna Bayesian Hyperparameter Search** — Compares Bi-LSTM with and without Attention.
5. **Monte Carlo Dropout** — Provides uncertainty bounds (confidence intervals) for predictions.
6. **Automated Export** — `.keras` model, `scaler.pkl`, and `metadata.json` for NeerNiti FastAPI backend deployment.

## 1. Environment Setup
Installs all required ML and forecasting libraries.

In [ ]:
# Install core forecasting and ML packages
!pip install -q tensorflow keras optuna prophet scikit-learn pandas numpy matplotlib joblib shap tqdm

import os
import sys
import json
import math
import warnings
from datetime import datetime, timezone
from typing import Any, Literal, Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from optuna.samplers import TPESampler
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Attention, Bidirectional, Concatenate, Dense, Dropout, Input, LSTM, multiply, Reshape
)
from tensorflow.keras.models import Model, Sequential

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Detect runtime environment
RUNTIME = 'local'
if 'google.colab' in sys.modules:
    RUNTIME = 'colab'
elif os.path.exists('/kaggle'):
    RUNTIME = 'kaggle'

print(f'Runtime: {RUNTIME.upper()}')
print(f'TensorFlow Version: {tf.__version__}')


## 2. Pipeline Configuration
Define sequence lengths, random seeds, and optimization bounds.

In [ ]:
SEQUENCE_LENGTH = 24                 # 24 months of history per forecast
RANDOM_SEED = 42
N_WALKFORWARD_SPLITS = 3             # Rolling-origin folds
WALKFORWARD_TEST_MONTHS = 12         # Size of each rolling test window
N_OPTUNA_TRIALS = 25
DEVIATION_THRESHOLD = 0.15           # Flags "Underperforming" if predicted vs actual differ by >15%
MC_DROPOUT_SAMPLES = 30              # Forward passes for Monte Carlo Dropout uncertainty estimation

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

VERSION_ID = f'verification_lstm_v2.0_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'
print(f'Version: {VERSION_ID}')
print(f'Walkforward config: {N_WALKFORWARD_SPLITS} folds of {WALKFORWARD_TEST_MONTHS} months each.')


## 3. Data Loading & Fallback Simulation
Since Colab cannot directly connect to a local PostgreSQL database, we provide two options:
1. **Option 1**: Generate simulated but hydrogeologically calibrated groundwater levels mimicking real Indian wells.
2. **Option 2**: Upload your own CSV extracted from PostGIS (`well_telemetry`).

In [ ]:
def generate_simulated_multi_well_data(num_wells: int = 6, num_months: int = 120) -> dict:
    """Simulates monthly groundwater + rainfall time series for several wells."""
    print(f"Simulating {num_months}-month telemetry for {num_wells} wells...")
    rng = np.random.default_rng(RANDOM_SEED)
    dates = pd.date_range(start="2016-01-01", periods=num_months, freq="MS")

    wells = {}
    for i in range(num_wells):
        well_id = f"WELL_{i:03d}"
        base_level = rng.uniform(6.0, 14.0)               # differing baseline depth per well
        depletion_rate = rng.uniform(0.02, 0.15)          # differing long-term decline severity
        seasonality_amp = rng.uniform(1.5, 2.5)

        months = np.array([d.month for d in dates])
        seasonality = seasonality_amp * np.sin(2 * np.pi * months / 12)
        trend = depletion_rate * np.arange(num_months)
        noise = rng.normal(0, 0.3, num_months)
        water_levels = base_level + seasonality + trend + noise

        rainfall = np.clip(
            120 + 100 * np.sin(2 * np.pi * (months - 6) / 12) + rng.normal(0, 15, num_months),
            0, None,
        )

        wells[well_id] = pd.DataFrame({"ds": dates, "y": water_levels, "rainfall_mm": rainfall})

    return wells

# Check if user uploaded a CSV, else use simulation
DATA_FILE = 'well_telemetry.csv'
wells = {}

if os.path.exists(DATA_FILE):
    print(f"Found {DATA_FILE}. Processing real data...")
    df = pd.read_csv(DATA_FILE)
    df['reading_date'] = pd.to_datetime(df['reading_date'])
    min_required_months = SEQUENCE_LENGTH + (WALKFORWARD_TEST_MONTHS * N_WALKFORWARD_SPLITS)
    
    for station, group in df.groupby('station_name'):
        group = group.set_index('reading_date').resample('MS').mean()
        group['water_level_m_bgl'] = group['water_level_m_bgl'].interpolate(method='linear')
        group = group.dropna().reset_index()
        
        if len(group) >= min_required_months:
            group.rename(columns={'reading_date': 'ds', 'water_level_m_bgl': 'y'}, inplace=True)
            # Synthetic seasonal monsoon proxy if rainfall isn't in CSV
            if 'rainfall_mm' not in group.columns:
                group['rainfall_mm'] = group['ds'].dt.month.map(
                    lambda m: np.random.uniform(200, 450) if m in [7, 8, 9] else np.random.uniform(0, 50)
                )
            wells[station] = group
    dataset_snapshot_name = f"Real_CSV_Well_Telemetry_v2.0_{len(wells)}wells"
else:
    wells = generate_simulated_multi_well_data(num_wells=8)
    dataset_snapshot_name = f"Simulated_MultiWell_Dataset_v2.0_{len(wells)}wells"

print(f"\nLoaded {len(wells)} valid wells for training.")
list(wells.values())[0].head()


## 4. Train Prophet Baseline Model
A standard Meta Prophet model acts as the baseline sanity check. We include rainfall as an exogenous regressor.

In [ ]:
def walk_forward_splits(df, n_splits, test_size):
    """Generates rolling-origin (walk-forward) train/test splits for time series."""
    splits = []
    total_len = len(df)
    min_train_size = total_len - n_splits * test_size
    if min_train_size < SEQUENCE_LENGTH * 2:
        raise ValueError(f"Not enough data for {n_splits} folds.")
    for fold in range(n_splits):
        train_end = min_train_size + fold * test_size
        test_end = train_end + test_size
        splits.append((df.iloc[:train_end].copy(), df.iloc[train_end:test_end].copy()))
    return splits

def train_baseline_prophet(train_df, test_df):
    """Trains a Prophet baseline (with rainfall as a regressor)."""
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    model.add_regressor("rainfall_mm")
    model.fit(train_df[["ds", "y", "rainfall_mm"]])

    future = model.make_future_dataframe(periods=len(test_df), freq="MS")
    full_rainfall = pd.concat([train_df["rainfall_mm"], test_df["rainfall_mm"]]).reset_index(drop=True)
    future["rainfall_mm"] = full_rainfall.values[: len(future)]

    forecast = model.predict(future)
    predictions = forecast["yhat"][-len(test_df):].values
    actuals = test_df["y"].values
    return float(np.sqrt(mean_squared_error(y_true=actuals, y_pred=predictions)))

print("Training baseline Prophet model per well...")
baseline_rmses = []
for well_id, df in wells.items():
    train_df, test_df = walk_forward_splits(df, n_splits=1, test_size=WALKFORWARD_TEST_MONTHS)[0]
    baseline_rmses.append(train_baseline_prophet(train_df, test_df))
    
baseline_rmse = float(np.mean(baseline_rmses))
print(f"Baseline Prophet mean RMSE across {len(wells)} wells: {baseline_rmse:.4f} meters")


## 5. Bidirectional LSTM + Attention (Architecture & Optuna Search)
Defines the deep learning models and optimizes hyperparameters using Bayesian search.

In [ ]:
def create_multivariate_windows(values, seq_length):
    X, y = [], []
    for i in range(len(values) - seq_length):
        X.append(values[i : i + seq_length])
        y.append(values[i + seq_length, 0])
    return np.array(X), np.array(y)

def build_bi_lstm_model(input_shape, units_1=64, units_2=32, dropout=0.2, learning_rate=0.001, use_attention=False):
    if not use_attention:
        model = Sequential([
            Bidirectional(LSTM(units_1, return_sequences=True), input_shape=input_shape),
            Dropout(dropout),
            Bidirectional(LSTM(units_2, return_sequences=False)),
            Dropout(dropout),
            Dense(16, activation="relu"),
            Dense(1, activation="linear"),
        ])
    else:
        inputs = Input(shape=input_shape)
        lstm_out = Bidirectional(LSTM(units_1, return_sequences=True))(inputs)
        lstm_out = Dropout(dropout)(lstm_out)

        attention_scores = Dense(1, activation="tanh")(lstm_out)
        attention_scores = Reshape((input_shape[0],))(attention_scores)
        attention_weights = Dense(input_shape[0], activation="softmax")(attention_scores)
        attention_weights = Reshape((input_shape[0], 1))(attention_weights)
        context_vector = multiply([lstm_out, attention_weights])
        context_vector = tf.keras.layers.Lambda(
            lambda x: tf.reduce_sum(x, axis=1), output_shape=(2 * units_1,)
        )(context_vector)

        dense = Dense(16, activation="relu")(context_vector)
        dense = Dropout(dropout)(dense)
        output = Dense(1, activation="linear")(dense)
        model = Model(inputs=inputs, outputs=output)

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model

def fit_and_evaluate_fold(train_df, test_df, params, seq_length=SEQUENCE_LENGTH, max_epochs=40, verbose=0):
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_train = scaler.fit_transform(train_df[["y", "rainfall_mm"]])
    combined = pd.concat([train_df[["y", "rainfall_mm"]].tail(seq_length), test_df[["y", "rainfall_mm"]]])
    scaled_test = scaler.transform(combined.values)

    X_train, y_train = create_multivariate_windows(scaled_train, seq_length)
    X_test, y_test = create_multivariate_windows(scaled_test, seq_length)

    model = build_bi_lstm_model(
        input_shape=(seq_length, 2),
        units_1=params.get("units_1", 64),
        units_2=params.get("units_2", 32),
        dropout=params.get("dropout", 0.2),
        learning_rate=params.get("learning_rate", 0.001),
        use_attention=params.get("use_attention", False),
    )

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6),
    ]

    model.fit(
        X_train, y_train, epochs=max_epochs, batch_size=params.get("batch_size", 16),
        validation_data=(X_test, y_test), callbacks=callbacks, verbose=verbose,
    )

    preds_scaled = model.predict(X_test, verbose=0).flatten()
    preds_full = np.column_stack([preds_scaled, np.zeros_like(preds_scaled)])
    actuals_full = np.column_stack([y_test, np.zeros_like(y_test)])
    preds_meters = scaler.inverse_transform(preds_full)[:, 0]
    actuals_meters = scaler.inverse_transform(actuals_full)[:, 0]

    rmse = float(np.sqrt(mean_squared_error(y_true=actuals_meters, y_pred=preds_meters)))
    return rmse, model, scaler

def make_objective(pooled_wells, use_attention):
    def objective(trial):
        params = {
            "units_1": trial.suggest_categorical("units_1", [32, 64, 96]),
            "units_2": trial.suggest_categorical("units_2", [16, 32, 64]),
            "dropout": trial.suggest_float("dropout", 0.1, 0.4),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32]),
            "use_attention": use_attention,
        }
        fold_rmses = []
        for well_id, df in pooled_wells.items():
            splits = walk_forward_splits(df, n_splits=1, test_size=WALKFORWARD_TEST_MONTHS)
            rmse, _, _ = fit_and_evaluate_fold(splits[0][0], splits[0][1], params, max_epochs=20, verbose=0)
            fold_rmses.append(rmse)
        return float(np.mean(fold_rmses))
    return objective

def tune_architecture(pooled_wells, use_attention, name):
    print(f"\nRunning Optuna Search for {name} ({N_OPTUNA_TRIALS} trials)...")
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED))
    study.optimize(make_objective(pooled_wells, use_attention), n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"  Best RMSE: {study.best_value:.4f} | Params: {study.best_params}")
    return study

plain_study = tune_architecture(wells, use_attention=False, name="Bi-LSTM (plain)")
attn_study = tune_architecture(wells, use_attention=True, name="Bi-LSTM + Attention")

if plain_study.best_value <= attn_study.best_value:
    WINNER_NAME = "Bi-LSTM"
    BEST_PARAMS = {**plain_study.best_params, "use_attention": False}
else:
    WINNER_NAME = "Bi-LSTM+Attention"
    BEST_PARAMS = {**attn_study.best_params, "use_attention": True}

print(f"\nWinner Architecture: {WINNER_NAME}")


## 6. Final Evaluation & Monte Carlo Dropout
Evaluates the winning architecture across all walkforward folds and calculates uncertainty bounds.

In [ ]:
print(f"Running full {N_WALKFORWARD_SPLITS}-fold walk-forward evaluation of {WINNER_NAME}...")
all_fold_rmses, all_fold_maes = [], []
final_model, final_scaler = None, None
last_train_df, last_test_df = None, None

for well_id, df in wells.items():
    splits = walk_forward_splits(df, n_splits=N_WALKFORWARD_SPLITS, test_size=WALKFORWARD_TEST_MONTHS)
    for train_df, test_df in splits:
        rmse, model, scaler = fit_and_evaluate_fold(train_df, test_df, BEST_PARAMS, max_epochs=50, verbose=0)
        all_fold_rmses.append(rmse)

        scaled_test = scaler.transform(
            pd.concat([train_df[["y", "rainfall_mm"]].tail(SEQUENCE_LENGTH), test_df[["y", "rainfall_mm"]]]).values
        )
        X_test, y_test = create_multivariate_windows(scaled_test, SEQUENCE_LENGTH)
        preds_scaled = model.predict(X_test, verbose=0).flatten()
        preds_full = np.column_stack([preds_scaled, np.zeros_like(preds_scaled)])
        actuals_full = np.column_stack([y_test, np.zeros_like(y_test)])
        preds_meters = scaler.inverse_transform(preds_full)[:, 0]
        actuals_meters = scaler.inverse_transform(actuals_full)[:, 0]
        
        all_fold_maes.append(float(mean_absolute_error(y_true=actuals_meters, y_pred=preds_meters)))
        
        final_model, final_scaler = model, scaler
        last_train_df, last_test_df = train_df, test_df

mean_rmse = float(np.mean(all_fold_rmses))
mean_mae = float(np.mean(all_fold_maes))
print(f"\n{WINNER_NAME} Mean Walk-Forward RMSE: {mean_rmse:.4f} meters")
print(f"{WINNER_NAME} Mean Walk-Forward MAE: {mean_mae:.4f} meters")

if mean_rmse < baseline_rmse:
    print(f"🏆 {WINNER_NAME} BEAT the Prophet baseline ({mean_rmse:.4f} < {baseline_rmse:.4f}).")
else:
    print(f"⚠️ {WINNER_NAME} did NOT beat baseline ({mean_rmse:.4f} >= {baseline_rmse:.4f}).")

# Monte Carlo Dropout
def mc_dropout_predict(model, X, n_samples=MC_DROPOUT_SAMPLES):
    preds = np.stack([model(X, training=True).numpy().flatten() for _ in range(n_samples)], axis=0)
    return preds.mean(axis=0), preds.std(axis=0)

scaled_last_test = final_scaler.transform(
    pd.concat([last_train_df[["y", "rainfall_mm"]].tail(SEQUENCE_LENGTH), last_test_df[["y", "rainfall_mm"]]]).values
)
X_last_test, _ = create_multivariate_windows(scaled_last_test, SEQUENCE_LENGTH)
mc_mean_scaled, mc_std_scaled = mc_dropout_predict(final_model, X_last_test)
scale_factor = 1.0 / final_scaler.scale_[0]
mc_std_meters = mc_std_scaled * scale_factor
print(f"Monte Carlo Dropout mean forecast uncertainty: +/-{mc_std_meters.mean():.3f} meters")


## 7. Export Artifacts for Deployment
Save the Keras model, scaler, and metadata JSON.

In [ ]:
model_path = f"{VERSION_ID}.keras"
scaler_path = f"{VERSION_ID}_scaler.pkl"
metadata_path = f"{VERSION_ID}_metadata.json"

final_model.save(model_path)
joblib.dump(final_scaler, scaler_path)

metadata = {
    "model_version_id": VERSION_ID,
    "module_name": "verification",
    "algorithm": WINNER_NAME,
    "trained_on": datetime.now(timezone.utc).isoformat(),
    "hyperparameters": {
        **BEST_PARAMS,
        "sequence_length": SEQUENCE_LENGTH,
        "optimizer": "Adam",
    },
    "evaluation_metrics": {
        "rmse": mean_rmse,
        "mae": mean_mae,
        "baseline_prophet_rmse": baseline_rmse,
        "mc_dropout_mean_uncertainty_m": float(mc_std_meters.mean()),
        "deviation_threshold": DEVIATION_THRESHOLD,
    },
    "training_data_snapshot": dataset_snapshot_name,
    'runtime': RUNTIME,
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

if RUNTIME == 'colab':
    print('\nDownloading artifacts to your local machine...')
    from google.colab import files
    for path in [model_path, scaler_path, metadata_path]:
        if os.path.exists(path):
            files.download(path)

print(f"\nArtifacts saved:\n- {model_path}\n- {scaler_path}\n- {metadata_path}")
print("\nUse register_colab_model.py in your backend to register this model.")
